<a href="https://colab.research.google.com/github/sklearner26/3d-models/blob/main/Hand_Gloves_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install Dependencies**

In [3]:
!pip install -q ultralytics opencv-python huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 7.3 MB/s eta 0:00:00


# **Install Datasets**

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rsaishivani/hand-glove-dataset")

print("Path to dataset files:", path)

100%|██████████| 70.1M/70.1M [00:02<00:00, 29.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/rsaishivani/hand-glove-dataset/versions/1


# **Move datasets to content folder**

In [2]:
import shutil
import os

source = "/root/.cache/kagglehub/datasets/rsaishivani/hand-glove-dataset/versions/1"
destination = "/content/hand-glove-dataset"

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Dataset copied to:", destination)
print(os.listdir(destination))

Dataset copied to: /content/hand-glove-dataset
['README.dataset.txt', 'data.yaml', 'valid', 'test', 'train', 'README.roboflow.txt']


# **install yolo model**

In [4]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Download pretrained model from Hugging Face
MODEL_PATH = hf_hub_download(
    repo_id="Hexmon/vyra-yolo-ppe-detection",
    filename="best.pt"
)

print("Model downloaded to:")
print(MODEL_PATH)

# Load local model
model = YOLO(MODEL_PATH)

print("\nModel loaded successfully!")
print("Classes:")
print(model.names)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


best.pt: reconstructing file:   0%|          |  0.00B / 52.1MB            

best.pt: downloading bytes:           |  0.00B            

Model downloaded to:
/root/.cache/huggingface/hub/models--Hexmon--vyra-yolo-ppe-detection/snapshots/08895b33d95d2587423ebe4f7c1b9c41beebd642/best.pt

Model loaded successfully!
Classes:
{0: 'Fall-Detected', 1: 'Gloves', 2: 'Goggles', 3: 'Hardhat', 4: 'Ladder', 5: 'Mask', 6: 'NO-Gloves', 7: 'NO-Goggles', 8: 'NO-Hardhat', 9: 'NO-Mask', 10: 'NO-Safety Vest', 11: 'Person', 12: 'Safety Cone', 13: 'Safety Vest'}


# **Fine tuning a model**

In [8]:
# Train model on hand glove dataset
from ultralytics import YOLO



model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/hand-glove-dataset/data.yaml",
    epochs=5,
    imgsz=416,
    batch=16,
    device=0,
    workers=2,
    project="/content/hand_glove_training",
    name="quick_test",
    patience=3
)

Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hand-glove-dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=quick_test, nbs=64, nms=

In [12]:
# save model
model.save("/content/hand_glove_best.pt")

In [15]:
# Evaluate on the TEST set
metrics = model.val(
    data="/content/hand-glove-dataset/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    conf=0.25,
    iou=0.5,
    device=0
)

Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1215.7±316.2 MB/s, size: 42.5 KB)
val: Scanning /content/hand-glove-dataset/test/labels... 108 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 108/108 2.2Kit/s 0.0s
val: New cache created: /content/hand-glove-dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.5it/s 2.8s
                   all        108        108      0.965      0.924      0.969      0.651
                 glove         54         54       0.93          1      0.993      0.594
              no_glove         54         54          1      0.848      0.945      0.708
Speed: 6.2ms preprocess, 6.5ms inference, 0.1ms loss, 3.4ms postprocess per image
Results saved to /content/runs/detect/val


In [16]:
print("\n========== MODEL EVALUATION ==========")

print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

print("======================================")


========== MODEL EVALUATION ==========
mAP@50: 0.9689090909090909
mAP@50-95: 0.6508470546113997
Precision: 0.9651395427948002
Recall: 0.9238054731878704


In [14]:
# test model on single image
from ultralytics import YOLO

MODEL_PATH = "/content/hand_glove_best.pt"

IMAGE_PATH = "/content/hand-glove-dataset/train/images/1596468248798_jpg.rf.2dd065d04b5a4c09518f63d700066c70.jpg"



results = model.predict(
    source=IMAGE_PATH,
    conf=0.25,
    save=True
)

for result in results:

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        x1, y1, x2, y2 = map(
            int,
            box.xyxy[0].tolist()
        )

        print({
            "class": model.names[class_id],
            "confidence": round(confidence, 4),
            "bbox": [x1, y1, x2, y2]
        })


image 1/1 /content/hand-glove-dataset/train/images/1596468248798_jpg.rf.2dd065d04b5a4c09518f63d700066c70.jpg: 416x416 1 glove, 9.3ms
Speed: 2.2ms preprocess, 9.3ms inference, 1.5ms postprocess per image at shape (1, 3, 416, 416)
Results saved to /content/runs/detect/predict-3
{'class': 'glove', 'confidence': 0.6943, 'bbox': [12, 78, 640, 489]}


In [13]:
# run test on all test data
import os
import cv2
import json
import glob
import torch

# ============================================================
# 2. CONFIGURATION
# ============================================================

INPUT_FOLDER = "/content/hand-glove-dataset/test/images"
OUTPUT_FOLDER = "/content/output_json"

CONF_THRESHOLD = 0.25

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# ============================================================
# 3. SHOW MODEL CLASSES
# ============================================================

print("\nModel classes:")

for class_id, class_name in model.names.items():
    print(f"{class_id}: {class_name}")


# ============================================================
# 4. FIND ALL TEST IMAGES
# ============================================================

extensions = (
    "*.jpg",
    "*.jpeg",
    "*.png",
    "*.bmp",
    "*.webp"
)

image_files = []

for extension in extensions:
    image_files.extend(
        glob.glob(
            os.path.join(
                INPUT_FOLDER,
                extension
            )
        )
    )

image_files.sort()

print(f"\nFound {len(image_files)} images.")


# ============================================================
# 5. PROCESS ALL IMAGES
# ============================================================

total_detections = 0

for index, image_path in enumerate(image_files, start=1):

    filename = os.path.basename(image_path)

    print(
        f"[{index}/{len(image_files)}] {filename}"
    )

    # --------------------------------------------------------
    # YOLO prediction
    # --------------------------------------------------------

    results = model.predict(
        source=image_path,
        conf=CONF_THRESHOLD,
        verbose=False
    )

    detections = []

    # --------------------------------------------------------
    # Process detections
    # --------------------------------------------------------

    for result in results:

        if result.boxes is None:
            continue

        for box in result.boxes:

            class_id = int(box.cls[0])
            confidence = float(box.conf[0])

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0].tolist()
            )

            class_name = model.names[class_id]

            # ------------------------------------------------
            # Convert your dataset labels
            # ------------------------------------------------

            name = class_name.lower().strip()

            if (
                "glove" in name
                and "no" not in name
            ):
                label = "gloved_hand"

            elif (
                "glove" in name
                and "no" in name
            ):
                label = "bare_hand"

            elif "bare" in name:
                label = "bare_hand"

            elif "gloved" in name:
                label = "gloved_hand"

            else:
                # Ignore unexpected classes
                continue

            # ------------------------------------------------
            # Create JSON detection
            # ------------------------------------------------

            detection = {
                "label": label,
                "confidence": round(
                    confidence,
                    4
                ),
                "bbox": [
                    x1,
                    y1,
                    x2,
                    y2
                ]
            }

            detections.append(detection)

    # --------------------------------------------------------
    # Save JSON
    # --------------------------------------------------------

    image_name = os.path.splitext(filename)[0]

    json_path = os.path.join(
        OUTPUT_FOLDER,
        image_name + ".json"
    )

    with open(json_path, "w") as f:

        json.dump(
            detections,
            f,
            indent=4
        )

    total_detections += len(detections)

    print(
        f"    → {len(detections)} detection(s)"
    )


# ============================================================
# 6. SUMMARY
# ============================================================

print("\n========================================")
print("Detection completed")
print("========================================")

print(f"Images processed : {len(image_files)}")
print(f"Total detections : {total_detections}")
print(f"JSON output      : {OUTPUT_FOLDER}")
print(f"Model weights    : {SAVE_PATH}")


Model classes:
0: glove
1: no_glove

Found 108 images.
[1/108] 20220225_103259_jpg.rf.365765f491604a7376af4e8cf00f6d8b.jpg
    → 1 detection(s)
[2/108] IMG20220224160807-1-_jpg.rf.0e6b5ef655e3ff4afbbd67f049b6a63b.jpg
    → 1 detection(s)
[3/108] IMG20220224160840_jpg.rf.354959c9070c5b86c661a533feae83a6.jpg
    → 1 detection(s)
[4/108] IMG20220224160928_jpg.rf.204ebb54112b265c2cecf3ff209ae236.jpg
    → 1 detection(s)
[5/108] IMG20220224162653_jpg.rf.d74c1b8b73da34dd9456cf46bc29a6c6.jpg
    → 1 detection(s)
[6/108] IMG20220224164705_jpg.rf.1f6ff0527f786e18ad04ffb61c3cb0f9.jpg
    → 1 detection(s)
[7/108] IMG20220224170028_jpg.rf.4957f65b4089c279ace77bbb3d34c691.jpg
    → 1 detection(s)
[8/108] IMG20220224172210_jpg.rf.893fde2d2ef4e33ab0acbab260c8d905.jpg
    → 1 detection(s)
[9/108] IMG20220224231707_jpg.rf.f60de50d501f2d9d61e34961d88da3a3.jpg
    → 1 detection(s)
[10/108] IMG20220224231747_jpg.rf.d535d78c22b438ce8fb8bece76f496c5.jpg
    → 1 detection(s)
[11/108] IMG20220224231810_jpg.r